### Create Age column based on DOB column (Age when crime was committed)

In [1]:
import pandas as pd 
import os

In [2]:
notebook_dir = os.getcwd()
data_dir = os.path.join(notebook_dir, '..', 'data')
checkpoint_dir = os.path.join(data_dir, 'checkpoints')

In [3]:
cp8 = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint8_combined_data.csv")

In [4]:
cp8

,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid,person_id,category,Year,crime_severity
0,25026764.0,2025-07-01 00:12:08,THREATS,259 ANDOVER ST,No,NaN,NaN,NaN,259 ANDOVER ST,"259 ANDOVER ST, Lawrence, MA",42.694842,-71.155948,10.0,2026-06-22T23:33:38.335777+00:00,True,NaN,Violent and Weapon Offenses,2025,Serious
1,25026765.0,2025-07-01 00:19:36,TOW/REPOSSED,454 HAMPSHIRE ST,No,NaN,NaN,NaN,454 HAMPSHIRE ST,"454 HAMPSHIRE ST, Lawrence, MA",42.717278,-71.171768,10.0,2026-06-25T23:39:24.046150+00:00,True,NaN,Motor Vehicle Incidents,2025,Non-Serious
2,25026766.0,2025-07-01 00:49:07,DISTURBANCE,155 HAVERHILL ST,No,NaN,NaN,NaN,155 HAVERHILL ST,"155 HAVERHILL ST, Lawrence, MA",42.710380,-71.161736,10.0,2026-06-19T16:04:18.901970+00:00,True,NaN,Public Disturbances,2025,Non-Serious
3,25026767.0,2025-07-01 01:15:01,CK WELL BEING,621 COMMON ST,No,NaN,NaN,NaN,621 COMMON ST,"621 COMMON ST, Lawrence, MA",42.705034,-71.169622,10.0,2026-06-26T19:04:44.576031+00:00,True,NaN,Medical and Welfare Assistance,2025,Non-Serious
4,25026769.0,2025-07-01 01:29:06,EXTRA SURVEIL,425 MERRIMACK ST,No,NaN,NaN,NaN,425 MERRIMACK ST,"425 MERRIMACK ST, Lawrence, MA",42.703240,-71.143607,10.0,2026-06-25T22:01:03.147567+00:00,True,NaN,Preventive Policing,2025,Non-Serious
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66736,25012595.0,2025-03-31 22:18:18,INVEST CONT,241 HOWARD ST,No,NaN,NaN,NaN,241 HOWARD ST,"241 HOWARD ST, Lawrence, MA",42.711757,-71.146320,10.0,2026-06-22T22:26:17.264637+00:00,True,NaN,Law Enforcement Operations,2025,Non-Serious
66737,25012596.0,2025-03-31 22:55:25,AUTO ACC/PI,LAWRENCE ST & PARK ST,No,NaN,NaN,NaN,LAWRENCE ST & PARK ST,"LAWRENCE ST & PARK ST, Lawrence, MA",42.715850,-71.173001,9.0,2026-06-29T05:27:22.730891+00:00,True,NaN,Motor Vehicle Incidents,2025,Non-Serious
66738,25012597.0,2025-03-31 23:05:23,LOCKOUT,582 ESSEX ST,No,NaN,NaN,NaN,582 ESSEX ST,"582 ESSEX ST, Lawrence, MA",42.704627,-71.167580,10.0,2026-06-26T18:04:20.701826+00:00,True,NaN,Motor Vehicle Incidents,2025,Non-Serious
66739,25012598.0,2025-03-31 23:25:25,ALARM/BURG,200 COMMON ST,No,NaN,NaN,NaN,200 COMMON ST,"200 COMMON ST, Lawrence, MA",42.708218,-71.159971,9.0,2026-06-20T19:58:44.903084+00:00,True,NaN,Fire and Arson Incidents,2025,Non-Serious


In [5]:
# work on a copy
df = cp8.copy()

# Optional, tidy column names and Arrested values
df.columns = df.columns.str.strip()
if 'Arrested' in df.columns:
    df['Arrested'] = df['Arrested'].astype(str).str.strip().str.title()

# Keep originals for debugging
df['_DOB_raw'] = df['DOB'].astype(str)
df['_Date_raw'] = df['Date'].astype(str)

# Parse to datetime
df['DOB_dt'] = pd.to_datetime(df['DOB'], errors='coerce', utc=False)
df['Date_dt'] = pd.to_datetime(df['Date'], errors='coerce', utc=False)

# Summary of parsing results
dob_total = df['DOB'].shape[0]
dob_nonnull = df['DOB'].notna().sum()
dob_failed = df['DOB_dt'].isna() & df['DOB'].notna()
date_failed = df['Date_dt'].isna() & df['Date'].notna()

print("DOB non-null count:", dob_nonnull)
print("DOB failed to parse count:", dob_failed.sum())
print("Date failed to parse count:", date_failed.sum())

# Show a few examples of bad DOB and Date entries
if dob_failed.any():
    print("\nSample problematic DOB values:")
    print(df.loc[dob_failed, '_DOB_raw'].drop_duplicates().head(20).to_list())

if date_failed.any():
    print("\nSample problematic Date values:")
    print(df.loc[date_failed, '_Date_raw'].drop_duplicates().head(20).to_list())

# If you want to enforce the standardized columns back into cp8
# only do this once you are happy with the parse results
cp8['DOB'] = df['DOB_dt']
cp8['Date'] = df['Date_dt']

# Final sanity check
print("\nDtypes after standardization:")
print(cp8[['DOB', 'Date']].dtypes)


DOB non-null count: 917
DOB failed to parse count: 0
Date failed to parse count: 0

Dtypes after standardization:
DOB     datetime64[ns]
Date    datetime64[ns]
dtype: object


In [6]:
import numpy as np

In [7]:
# mask for rows we (those who were charged, assume charged if DOB is present)
mask_charged = cp8['DOB'].notna()

# compute age at offense date with birthday adjustment
before_birthday = (
    (cp8.loc[mask_charged, 'Date'].dt.month < cp8.loc[mask_charged, 'DOB'].dt.month) |
    (
        (cp8.loc[mask_charged, 'Date'].dt.month == cp8.loc[mask_charged, 'DOB'].dt.month) &
        (cp8.loc[mask_charged, 'Date'].dt.day   <  cp8.loc[mask_charged, 'DOB'].dt.day)
    )
)

In [8]:
print(before_birthday)

47        True
57       False
66       False
102      False
157      False
         ...  
66388     True
66453     True
66594     True
66696     True
66732    False
Length: 917, dtype: bool


In [9]:
age_years = (
    cp8.loc[mask_charged, 'Date'].dt.year - cp8.loc[mask_charged, 'DOB'].dt.year
    - before_birthday.astype(int)
)

In [10]:
print(age_years)

47       41
57       40
66       35
102      62
157      36
         ..
66388    33
66453    42
66594    24
66696    36
66732    29
Length: 917, dtype: int64


In [11]:
# Write Age, NaN where not charged per your rule
cp8['Age'] = np.nan
cp8.loc[mask_charged, 'Age'] = age_years

In [12]:
# Quick peek
print(cp8[['DOB','Date','Age']].head(10))
print(cp8['Age'].describe())


  DOB                Date  Age
0 NaT 2025-07-01 00:12:08  NaN
1 NaT 2025-07-01 00:19:36  NaN
2 NaT 2025-07-01 00:49:07  NaN
3 NaT 2025-07-01 01:15:01  NaN
4 NaT 2025-07-01 01:29:06  NaN
5 NaT 2025-07-01 01:29:08  NaN
6 NaT 2025-07-01 01:32:08  NaN
7 NaT 2025-07-01 01:33:05  NaN
8 NaT 2025-07-01 01:33:53  NaN
9 NaT 2025-07-01 01:38:35  NaN
count    917.000000
mean      36.369684
std       10.171761
min       18.000000
25%       29.000000
50%       36.000000
75%       42.000000
max       74.000000
Name: Age, dtype: float64


In [13]:
valid_age_df = cp8.loc[cp8['Age'].notna(), ['DOB', 'Date', 'Age']]

print(valid_age_df.head(20))   # just a preview

            DOB                Date   Age
47   1983-10-16 2025-07-01 09:27:27  41.0
57   1985-03-13 2025-07-01 11:01:23  40.0
66   1990-01-16 2025-07-01 12:15:49  35.0
102  1963-05-30 2025-07-01 17:08:48  62.0
157  1989-04-29 2025-07-02 02:28:19  36.0
163  1996-09-08 2025-07-02 07:24:16  28.0
186  1988-10-30 2025-07-02 11:01:01  36.0
230  1989-06-02 2025-07-02 17:53:06  36.0
373  1965-10-14 2025-07-03 19:11:40  59.0
402  1995-03-07 2025-07-03 23:21:27  30.0
445  2001-09-28 2025-07-04 02:34:17  23.0
683  1975-08-22 2025-07-05 04:34:36  49.0
704  1989-05-17 2025-07-05 10:55:00  36.0
1020 1994-05-27 2025-07-07 10:49:49  31.0
1138 2005-11-21 2025-07-08 06:08:43  19.0
1151 1985-04-26 2025-07-08 09:10:20  40.0
1175 1994-04-02 2025-07-08 12:00:34  31.0
1281 1991-12-08 2025-07-09 11:03:33  33.0
1310 1984-07-16 2025-07-09 14:51:08  40.0
1586 1987-05-23 2025-07-11 10:08:26  38.0


In [14]:
save_path = os.path.join(
    notebook_dir,
    "data",
    "missing_dates_csv",
    "md_checkpoints",
    "checkpoint9_age.csv"
)

cp8.to_csv(save_path, index=False)
print(f"Saved with Age column to: {save_path}")

Saved with Age column to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint9_age.csv
